In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

In [ ]:
# Loading Iris Dataset

iris = load_iris()
X = iris.data
y = iris.target

X_train,X_test,y_train,y_test = train_test_split(X,y,train_size=0.8,random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(X_train.shape)
print(X_test.shape)

(120, 4)
(30, 4)


In [9]:
class GDA:
    def __init__(self):
        self.classes = None
        self.priors = {}
        self.means = {}
        self.covariances = None
    
    def fit(self,X,y):
        m,n = X.shape
        self.classes = np.unique(y)
        self.covariances = np.zeros((n,n))
        for cls in self.classes:
            X_cls = X[y==cls]
            
            # Setting prior
            self.priors[cls] = len(X_cls)/m
            
            # Setting mean
            self.means[cls]=np.mean(X_cls,axis=0)
            
            # Setting covariance
            diff = X_cls-self.means[cls]
            self.covariances += diff.T@diff
        self.covariances/=m
    
    def gaussian_pdf_log(self,x,avg):
        n = len(avg)
        cov_inv = np.linalg.inv(self.covariances)
        cov_det = np.linalg.det(self.covariances)        
        diff = x-avg
        # Normalisation Constant doesn't matter
        log1 = -0.5*np.log(cov_det)
        log2 = -0.5*(diff.T@cov_inv@diff)
        return (log1 + log2)
    
    def predict(self,X):
        predictions = []
        for x in X:
            scores = []
            for cls in self.classes:
                prior = np.log(self.priors[cls])
                ll = self.gaussian_pdf_log(x,self.means[cls])
                score = prior + ll
                scores.append(score)
            predictions.append(np.argmax(scores))
        return np.array(predictions)            

In [12]:
model = GDA()
model.fit(X_train,y_train)

y_pred = model.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test,y_pred)}")

Accuracy : 1.0
